# 01 — Exploratory Data Analysis

Analyses the AQI feature data using **functions from `src/pearls_aqi`** (no
duplicated production logic). Run `python scripts/seed_sample_data.py` first if
the feature store is empty.

> ⚠️ If the store holds **sample** data (`data_source = sample`) these findings
> describe *synthetic* data and must not be read as real air quality. Re-run on
> backfilled production data for real insights. Charts are saved to
> `artifacts/eda/` as HTML (no extra dependencies needed).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.io as pio

from pearls_aqi.config import load_config
from pearls_aqi.storage import get_feature_store
from pearls_aqi.storage.base import FEATURES_GROUP

OUT = Path("artifacts/eda"); OUT.mkdir(parents=True, exist_ok=True)
cfg = load_config()
loc = cfg.active_location
store = get_feature_store(cfg)
df = store.read_features(FEATURES_GROUP, city_id=loc.city_id)
print(f"Loaded {len(df):,} rows for {loc.city}; source(s):", df['data_source'].unique() if 'data_source' in df else 'n/a')
df[["timestamp", "aqi", "pm2_5", "pm10", "temperature", "wind_speed"]].head()

In [ ]:
# AQI + pollutant summary
df[["aqi", "pm2_5", "pm10", "o3", "no2", "so2", "co", "temperature", "humidity", "wind_speed"]].describe().round(1)

In [ ]:
# Missing-data overview (uses the production validator)
from pearls_aqi.data.validation import missing_data_report
missing_data_report(df).head(15)

In [ ]:
# AQI distribution
fig = px.histogram(df, x="aqi", nbins=50, title=f"AQI distribution — {loc.city}")
fig.write_html(OUT / "aqi_distribution.html"); fig.show()

In [ ]:
# Diurnal (hour-of-day) and monthly patterns
df["hour"] = pd.to_datetime(df["timestamp"], utc=True).dt.hour
df["month"] = pd.to_datetime(df["timestamp"], utc=True).dt.month
hourly = df.groupby("hour")["aqi"].mean().reset_index()
monthly = df.groupby("month")["aqi"].mean().reset_index()
fig = px.line(hourly, x="hour", y="aqi", markers=True, title="Mean AQI by hour of day")
fig.write_html(OUT / "aqi_by_hour.html"); fig.show()
px.line(monthly, x="month", y="aqi", markers=True, title="Mean AQI by month").show()

In [ ]:
# Weekend vs weekday
df["is_weekend"] = pd.to_datetime(df["timestamp"], utc=True).dt.dayofweek >= 5
df.groupby("is_weekend")["aqi"].agg(["mean", "median", "count"]).round(1)

In [ ]:
# Weather / pollutant correlations with AQI
cols = ["aqi", "pm2_5", "pm10", "o3", "no2", "temperature", "humidity", "wind_speed", "pressure"]
corr = df[cols].corr().round(2)
fig = px.imshow(corr, text_auto=True, aspect="auto", title="Correlation matrix")
fig.write_html(OUT / "correlations.html"); fig.show()
print("Correlation of weather with AQI:\n", corr["aqi"].sort_values())

In [ ]:
# AQI autocorrelation (persistence structure)
s = df.sort_values("timestamp")["aqi"].reset_index(drop=True)
acf = {lag: round(s.autocorr(lag), 3) for lag in [1, 3, 6, 12, 24, 48, 72]}
print("AQI autocorrelation by lag (hours):", acf)

## EDA summary (sample data)

- AQI is **PM2.5-dominated**, with the expected **diurnal** (rush-hour) peaks and
  **seasonal** (winter-worse) structure built into the generator.
- AQI shows strong short-lag **autocorrelation** (persistence), decaying with lag
  — which is why persistence/rolling baselines are non-trivial to beat at short
  horizons and lag/rolling features are informative.
- Wind speed correlates **negatively** with AQI (dispersion); humidity/temperature
  show weaker relationships.

_These observations are on synthetic sample data. Re-run on production data for
real conclusions._